# Notebook 01 — OpenI Data Loading & QA Dataset Creation

This notebook:
1. Downloads the OpenI chest X-ray dataset (public, no registration needed)
2. Parses XML radiology reports
3. Generates a QA dataset using Groq LLaMA 3.1 8B Instant

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q groq python-dotenv tqdm pandas

In [ ]:
import os, sys
# Mount Google Drive to persist data between Colab sessions
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cxr_rag'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)

In [ ]:
# ── Download OpenI dataset ────────────────────────────────────────────────────
IMAGES_ZIP  = '/content/openi_images.zip'
REPORTS_TGZ = '/content/openi_reports.tgz'
IMAGES_DIR  = '/content/openi/images'
REPORTS_DIR = '/content/openi/reports'

import subprocess
if not os.path.exists(IMAGES_DIR):
    print('Downloading OpenI images (~1 GB) ...')
    subprocess.run(['wget', '-q', 'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.zip',
                    '-O', IMAGES_ZIP], check=True)
    subprocess.run(['unzip', '-q', IMAGES_ZIP, '-d', '/content/openi/'], check=True)

if not os.path.exists(REPORTS_DIR):
    print('Downloading OpenI reports (~20 MB) ...')
    subprocess.run(['wget', '-q', 'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz',
                    '-O', REPORTS_TGZ], check=True)
    os.makedirs(REPORTS_DIR, exist_ok=True)
    subprocess.run(['tar', '-xzf', REPORTS_TGZ, '-C', REPORTS_DIR], check=True)

print('Images dir:', os.listdir('/content/openi/')[:5])

In [ ]:
# ── Clone project repo ────────────────────────────────────────────────────────
# Replace with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/cxr-rag-system.git'
!git clone {REPO_URL} /content/cxr-rag-system
sys.path.insert(0, '/content/cxr-rag-system')

In [ ]:
# ── Parse OpenI XML reports ───────────────────────────────────────────────────
from src.data.openi_loader import OpenILoader

# OpenI extracts to ecgen-radiology/ inside the tar archive
reports_subdir = os.path.join(REPORTS_DIR, 'ecgen-radiology')
if not os.path.exists(reports_subdir):
    reports_subdir = REPORTS_DIR  # fallback

loader = OpenILoader(reports_dir=reports_subdir, images_dir=IMAGES_DIR)
df = loader.load()
print(f'Loaded {len(df)} studies with valid impression + frontal image')
df.head(3)

In [ ]:
# ── Train/val/test split ──────────────────────────────────────────────────────
train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

import pandas as pd
os.makedirs('/content/cxr-rag-system/data/processed', exist_ok=True)
full_df.to_csv('/content/cxr-rag-system/data/processed/reports_corpus.csv', index=False)
# Copy to Drive for persistence
import shutil
shutil.copy('/content/cxr-rag-system/data/processed/reports_corpus.csv',
            os.path.join(DRIVE_ROOT, 'reports_corpus.csv'))
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
# ── Configure Groq API ────────────────────────────────────────────────────────
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')  # set this in Colab Secrets (key icon)

from src.data.qa_creator import QACreator
creator = QACreator(groq_api_key=GROQ_API_KEY)

In [ ]:
# ── Generate QA dataset ───────────────────────────────────────────────────────
# Start with train split, then run val/test separately if time allows
# ~3,000 studies × ~6 questions = ~18,000 QA pairs
# At 28 req/min → ~10-11 hours for full train set
# For a demo: max_studies=500 → ~30 min

QA_OUTPUT = '/content/cxr-rag-system/data/processed/qa_dataset.jsonl'

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=500,  # increase to None for full dataset (takes hours)
)

print(f'\nGenerated {len(pairs)} QA pairs')
shutil.copy(QA_OUTPUT, os.path.join(DRIVE_ROOT, 'qa_dataset.jsonl'))

In [ ]:
# ── Inspect sample QA pairs ───────────────────────────────────────────────────
import json
with open(QA_OUTPUT) as f:
    samples = [json.loads(l) for l in f][:5]

for s in samples:
    print(f"Category : {s['category']}")
    print(f"Q        : {s['question']}")
    print(f"A        : {s['answer']}")
    print()

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
import pandas as pd
qa_df = pd.read_json(QA_OUTPUT, lines=True)
print('Category distribution:')
print(qa_df['category'].value_counts())
print(f'\nTotal pairs: {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'Split distribution:\n{qa_df["split"].value_counts()}')